# W&B Sweep — Actor-Critic (AC)
Búsqueda de hiperparámetros para el agente AC en el ambiente Simple (CSTR).
- Método: Random Search
- Proyecto W&B: `Tesis_AC_CTRL`
- Arquitectura: simple (solo CTRL)
- Acción: continua

## 1. Instalación e Imports

In [ ]:
import os
import random
import numpy as np
import torch
import wandb
import sys
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Clonar desde Github:
!git clone https://github.com/valeriaeskenazi/Control-PID-Adaptativo-Inteligente-mediante-Reinforcement-Learning.git
PROJECT_PATH = '/content/Control-PID-Adaptativo-Inteligente-mediante-Reinforcement-Learning/Version_4'
sys.path.append(PROJECT_PATH)

In [ ]:
from Environment.Simulation_Env.Reactor_CSTR import CSTRSimulator
from Environment.PIDControlEnv_simple import PIDControlEnv_Simple
from Environment.PIDControlEnv_complex import PIDControlEnv_Complex
from Agente.AC.train_AC import ACTrainer
from Agente.AC.algorithm_AC import ACAgent
from Aux.Plots import SimplePlotter, print_summary

print('Imports completados')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)

## 2. Login W&B

In [ ]:
!pip install wandb --quiet

In [ ]:
wandb.login()

## 3. Configuración del Sweep

In [ ]:
# ============ LISTAS DE OPCIONES PREDEFINIDAS ============

REWARD_WEIGHTS_OPTIONS = [
    {'error': 1.0, 'tiempo': 0.3,   'overshoot': 0.2,  'energy': 0.1},    # 0: balanceado
    {'error': 2.0, 'tiempo': 0.1,   'overshoot': 0.5,  'energy': 0.1},    # 1: foco en error y overshoot
    {'error': 1.0, 'tiempo': 0.001, 'overshoot': 0.3,  'energy': 0.001},  # 2: config base
    {'error': 3.0, 'tiempo': 0.1,   'overshoot': 0.1,  'energy': 0.05},   # 3: solo error importa
]

HIDDEN_DIMS_OPTIONS = [
    (64, 32),
    (128, 64),
    (128, 128, 64),
    (256, 128, 64),
]

# ============ FIJOS PARA TODOS LOS RUNS ============
SEED                         = 42
N_EPISODES                   = 300
EVAL_FREQUENCY               = 50
EARLY_STOPPING_PATIENCE      = 10
EARLY_STOPPING_MIN_DELTA_PCT = 0.01
N_MANIPULABLE_VARS           = 2
MANIPULABLE_RANGES           = [(300, 420), (99.5, 104)]
DT                           = 1.0
DEVICE                       = 'cuda' if torch.cuda.is_available() else 'cpu'

WANDB_TEAM    = 've326684-universidad-ort-uruguay'
WANDB_PROJECT = 'Tesis_AC_CTRL'

sweep_config = {
    'name':   'ac_cstr_random_search',
    'method': 'random',

    'metric': {
        'name': 'eval_reward',
        'goal': 'maximize'
    },

    'parameters': {

        # ============ AMBIENTE ============
        'max_time_detector':  {'values': [15, 30, 60]},
        'max_steps':          {'values': [20, 50, 100]},
        'reward_dead_band':   {'values': [0.01, 0.02, 0.05]},
        'delta_percent_ctrl': {'values': [0.05, 0.1, 0.2, 0.3]},
        'reward_weights_idx': {'values': [0, 1, 2, 3]},

        # ============ CRITERIOS DE ESTABILIDAD ============
        'error_increase_tolerance': {'values': [1.2, 1.5, 2.0]},
        'max_sign_changes_ratio':   {'values': [0.1, 0.2, 0.3]},
        'max_abrupt_change_ratio':  {'values': [0.03, 0.05, 0.1]},
        'abrupt_change_threshold':  {'values': [0.2, 0.3, 0.5]},

        # ============ AGENTE AC ============
        'hidden_dims_idx': {'values': [0, 1, 2, 3]},
        'lr_actor':        {'values': [0.00001, 0.0001, 0.0003]},
        'lr_critic':       {'values': [0.0001,  0.001,  0.01]},
        'gamma':           {'values': [0.95, 0.99, 0.999]},
        'entropy_coef':    {'values': [0.001, 0.01, 0.05]},
        'batch_size':      {'values': [32, 64, 128]},
        'buffer_size':     {'values': [10000, 50000, 100000]},
        'warmup_steps':    {'values': [200, 500, 1000]},
    }
}

sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT, entity=WANDB_TEAM)
print(f'Sweep creado: {sweep_id}')

## 4. Función del Sweep

In [ ]:
def sweep_run():
    # -------- Inicializar run --------
    wandb.init()
    cfg = wandb.config

    # -------- Reproducibilidad --------
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False
    wandb.config.update({'seed': SEED}, allow_val_change=True)

    # -------- Resolver índices --------
    reward_weights = REWARD_WEIGHTS_OPTIONS[cfg.reward_weights_idx]
    hidden_dims    = HIDDEN_DIMS_OPTIONS[cfg.hidden_dims_idx]

    wandb.config.update({
        'reward_weights': str(reward_weights),
        'hidden_dims':    str(hidden_dims),
    }, allow_val_change=True)

    # -------- Configurar CSTR --------
    cstr = CSTRSimulator(
        dt=DT,
        control_limits=(MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1])
    )

    # -------- Construir config del trainer --------
    trainer_config = {
        # === AMBIENTE ===
        'env_config': {
            'architecture':          'simple',
            'env_type':              'simulation',
            'action_type':           'continuous',
            'n_manipulable_vars':    N_MANIPULABLE_VARS,
            'manipulable_ranges':    MANIPULABLE_RANGES,
            'manipulable_setpoints': None,
            'dt_usuario':            DT,
            'max_steps':             cfg.max_steps,
            'max_time_detector':     cfg.max_time_detector,
            'reward_dead_band':      cfg.reward_dead_band,
            'delta_percent_ctrl':    cfg.delta_percent_ctrl,
            'reward_weights':        reward_weights,
            'pid_limits': [
                (0.01, 50.0),
                (0.001, 1.0),
                (0.0,   1.0)
            ],
            'agent_controller_config': {'agent_type': 'continuous'},
            'env_type_config': {
                'dt': DT,
                'control_limits': (MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1])
            },
            'stability_config': {
                'error_increase_tolerance': cfg.error_increase_tolerance,
                'max_sign_changes_ratio':   cfg.max_sign_changes_ratio,
                'max_abrupt_change_ratio':  cfg.max_abrupt_change_ratio,
                'abrupt_change_threshold':  cfg.abrupt_change_threshold,
            },
        },

        # === AGENTE AC ===
        'agent_ctrl_config': {
            'algorithm':    'ac',
            'state_dim':    N_MANIPULABLE_VARS * 5,
            'action_dim':   N_MANIPULABLE_VARS * 3,
            'n_vars':       N_MANIPULABLE_VARS,
            'action_type':  'continuous',
            'hidden_dims':  hidden_dims,
            'lr_actor':     cfg.lr_actor,
            'lr_critic':    cfg.lr_critic,
            'gamma':        cfg.gamma,
            'entropy_coef': cfg.entropy_coef,
            'batch_size':   cfg.batch_size,
            'buffer_size':  cfg.buffer_size,
            'warmup_steps': cfg.warmup_steps,
            'device':       DEVICE,
            'seed':         SEED,
        },

        # === ENTRENAMIENTO ===
        'n_episodes':                    N_EPISODES,
        'eval_frequency':                EVAL_FREQUENCY,
        'save_frequency':                9999,
        'log_frequency':                 50,
        'checkpoint_dir':                f'checkpoints/ac_{wandb.run.name}',
        'early_stopping_patience':       EARLY_STOPPING_PATIENCE,
        'early_stopping_min_delta_pct':  EARLY_STOPPING_MIN_DELTA_PCT,
        'use_wandb': True,
    }

    # -------- Conectar CSTR al ambiente --------
    trainer = ACTrainer(trainer_config)
    trainer.env.proceso.connect_external_process(cstr)

    # -------- Entrenar --------
    trainer.train()

    # -------- Métricas finales del run --------
    wandb.log({
        'final_eval_reward':      trainer.best_reward,
        'total_episodes':         len(trainer.episode_rewards),
        'final_reward_mean10':    np.mean(trainer.episode_rewards[-10:]),
        'final_energy_mean10':    np.mean(trainer.episode_energies[-10:]),
        'final_overshoot_mean10': np.mean(trainer.episode_max_overshoots[-10:]),
        'final_actor_loss_mean10':  np.mean(trainer.actor_losses[-10:])  if trainer.actor_losses  else 0,
        'final_critic_loss_mean10': np.mean(trainer.critic_losses[-10:]) if trainer.critic_losses else 0,
    })

    print(f'Run completado: {wandb.run.name}')
    wandb.finish()

## 5. Lanzar Sweep

In [ ]:
wandb.agent(sweep_id, function=sweep_run, count=30)

## 6. Run de Producción (post-sweep)
Completar con los mejores hiperparámetros encontrados en el sweep.

In [ ]:
# ============ CONFIG RUN PRINCIPAL AC 15K ============

WANDB_ENTITY  = 've326684-universidad-ort-uruguay'
WANDB_PROJECT = 'Tesis_AC_CTRL_PROD'
RUN_NAME      = 'ac_ctrl_15k_prod'

N_EPISODES                   = 15000
EVAL_FREQUENCY               = 100
LOG_FREQUENCY                = 100
SAVE_FREQUENCY               = 2000
EARLY_STOPPING_PATIENCE      = 40
EARLY_STOPPING_MIN_DELTA_PCT = 0.01
SEED                         = 42

trainer_config = {
    'env_config': {
        'architecture'           : 'simple',
        'env_type'               : 'simulation',
        'action_type'            : 'continuous',
        'n_manipulable_vars'     : 2,
        'manipulable_ranges'     : [(300, 420), (99.5, 104)],
        'manipulable_setpoints'  : None,
        'dt_usuario'             : 1.0,
        'max_steps'              : 100,        # ← ajustar con resultado del sweep
        'max_time_detector'      : 60,         # ← ajustar con resultado del sweep
        'reward_dead_band'       : 0.02,       # ← ajustar con resultado del sweep
        'delta_percent_ctrl'     : 0.2,        # ← ajustar con resultado del sweep
        'reward_weights'         : {'error': 1.0, 'tiempo': 0.001, 'overshoot': 0.3, 'energy': 0.001},
        'pid_limits'             : [(0.01, 50.0), (0.001, 1.0), (0.0, 1.0)],
        'agent_controller_config': {'agent_type': 'continuous'},
        'env_type_config'        : {'dt': 1.0, 'control_limits': ((300, 420), (99.5, 104))},
        'stability_config'       : {
            'error_increase_tolerance': 2.0,
            'max_sign_changes_ratio'  : 0.3,
            'max_abrupt_change_ratio' : 0.05,
            'abrupt_change_threshold' : 0.2,
        },
    },
    'agent_ctrl_config': {
        'algorithm'    : 'ac',
        'state_dim'    : 10,
        'action_dim'   : 6,
        'n_vars'       : 2,
        'action_type'  : 'continuous',
        'hidden_dims'  : (64, 32),       # ← ajustar con resultado del sweep
        'lr_actor'     : 1e-05,          # ← ajustar con resultado del sweep
        'lr_critic'    : 1e-03,          # ← ajustar con resultado del sweep
        'gamma'        : 0.95,           # ← ajustar con resultado del sweep
        'entropy_coef' : 0.01,           # ← ajustar con resultado del sweep
        'batch_size'   : 64,             # ← ajustar con resultado del sweep
        'buffer_size'  : 50000,          # ← ajustar con resultado del sweep
        'warmup_steps' : 500,            # ← ajustar con resultado del sweep
        'device'       : 'cuda' if torch.cuda.is_available() else 'cpu',
        'seed'         : SEED,
    },
    'n_episodes'                  : N_EPISODES,
    'eval_frequency'              : EVAL_FREQUENCY,
    'log_frequency'               : LOG_FREQUENCY,
    'save_frequency'              : SAVE_FREQUENCY,
    'checkpoint_dir'              : f'checkpoints/{RUN_NAME}',
    'early_stopping_patience'     : EARLY_STOPPING_PATIENCE,
    'early_stopping_min_delta_pct': EARLY_STOPPING_MIN_DELTA_PCT,
    'use_wandb': True,
}

# ============ REPRODUCIBILIDAD ============
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

# ============ INIT WANDB ============
wandb.init(
    project = WANDB_PROJECT,
    entity  = WANDB_ENTITY,
    name    = RUN_NAME,
    tags    = ['ac', '15k', 'produccion'],
    config  = trainer_config,
)

# ============ ENTRENAR ============
cstr    = CSTRSimulator(dt=1.0, control_limits=((300, 420), (99.5, 104)))
trainer = ACTrainer(trainer_config)
trainer.env.proceso.connect_external_process(cstr)
trainer.train()

# ============ MÉTRICAS FINALES ============
wandb.log({
    'final_eval_reward'       : trainer.best_reward,
    'total_episodes'          : len(trainer.episode_rewards),
    'final_reward_mean10'     : np.mean(trainer.episode_rewards[-10:]),
    'final_energy_mean10'     : np.mean(trainer.episode_energies[-10:]),
    'final_overshoot_mean10'  : np.mean(trainer.episode_max_overshoots[-10:]),
    'final_actor_loss_mean10' : np.mean(trainer.actor_losses[-10:])  if trainer.actor_losses  else 0,
    'final_critic_loss_mean10': np.mean(trainer.critic_losses[-10:]) if trainer.critic_losses else 0,
}, step=len(trainer.episode_rewards))

wandb.finish()
print(f'Run completado: {RUN_NAME}')